# Benchmark 探索

`analysis/data/sample_benchmark_runs.jsonl`(固定サンプル)で完結する。
実データを見るには先に `python -m analysis.export benchmark_runs analysis/data/benchmark_runs.jsonl` で
エクスポートし、下の `PATH` を差し替える。

In [ ]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # nbconvert / CI 用。対話実行なら不要

from analysis.benchmark_report import by_algorithm, input_size_curve
from analysis.loaders import load_benchmark_runs
from analysis.plots import plot_comparison, plot_input_size_curve

PATH = Path("..") / "data" / "sample_benchmark_runs.jsonl"
df = load_benchmark_runs(PATH)
df.head()

## アルゴリズム別の中央値

In [ ]:
report = by_algorithm(df)
report

In [ ]:
fig = plot_comparison(report, metric="elapsed_ms_median")
fig

## 入力サイズ別カーブ

サンプルには `size` 列が無い。実運用では複数サイズで benchmark を回し、各行に問題サイズを
付けた DataFrame を `input_size_curve` に渡す。ここでは擬似的に nodes 数から `size` を作る。

In [ ]:
import json

records = [json.loads(row) for row in PATH.read_text().splitlines() if row.strip()]
sizes = {r["id"]: len(r["payload"]["problem"]["data"]["nodes"]) for r in records}
df["size"] = df["benchmark_id"].map(sizes)
curve = input_size_curve(df)
curve

In [ ]:
plot_input_size_curve(curve, log=True)